# EEG Analysis: C9orf72-KO vs WT — Longitudinal Study

**Study design:**

| Age | Recording type | Analysis |
|-----|---------------|----------|
| 3 months | 24h baseline EEG | Per-animal band power |
| 4 months | 4h post kainic acid injection | Seizure-associated discharge detection |
| 6 months | 24h follow-up EEG | Per-animal band power |
| 12 months | 24h follow-up EEG | Per-animal band power |

**Biological question:** Do C9orf72-KO mice show progressive changes in network activity compared to WT controls across disease progression?

**Key finding:** C9orf72-KO mice show a U-shaped elevation in theta band power (4–8 Hz) at 3 and 12 months, with convergence at 6 months, suggesting early network dysfunction, transient compensation, and re-emergence of dysfunction at late disease stage.

---

## 0. Setup

In [ ]:
import os
import sys
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyabf
from scipy.stats import mannwhitneyu

sys.path.insert(0, os.path.join("..", "src"))

from preprocessing import load_abf, remove_artifacts, estimate_baseline
from detection import detect_discharges, compute_psd, compute_band_power
from classify import build_feature_matrix, train_classifier, evaluate_classifier, plot_feature_importance, plot_roc_and_pr

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
print("Environment ready.")

## 1. Data paths

In [ ]:
BASE = r"C:\Users\belay\OneDrive\Desktop\EEG analysis"

PATHS = {
    "3m": {
        "WT": os.path.join(BASE, "Baseline EEG", "WT"),
        "KO": os.path.join(BASE, "Baseline EEG", "KO"),
        "manifest_wt": "abf_files_start_times_wt.xlsx",
        "manifest_ko": "abf_files_start_times_ko.xlsx",
    },
    "4m_KA": {
        "WT": os.path.join(BASE, "KA EEG", "WT"),
        "KO": os.path.join(BASE, "KA EEG", "KO"),
    },
    "6m": {
        "WT": os.path.join(BASE, "Six Months EEG", "WT"),
        "KO": os.path.join(BASE, "Six Months EEG", "KO"),
        "manifest_wt": "abf_files_start_times_wt_6m.xlsx",
        "manifest_ko": "abf_files_start_times_ko_6m.xlsx",
    },
    "12m": {
        "WT": os.path.join(BASE, "One Year EEG", "WT"),
        "KO": os.path.join(BASE, "One Year EEG", "KO"),
        "manifest_wt": "abf_files_start_times_wt_1y.xlsx",
        "manifest_ko": "abf_files_start_times_ko_1y.xlsx",
    },
}

print("Checking paths:")
for timepoint, dirs in PATHS.items():
    wt_ok = os.path.exists(dirs["WT"])
    ko_ok = os.path.exists(dirs["KO"])
    print(f"  {timepoint}: WT={'OK' if wt_ok else 'NOT FOUND'} | KO={'OK' if ko_ok else 'NOT FOUND'}")

## 2. Helper functions

In [ ]:
def build_file_index(data_dir):
    """Recursively find all ABF files in a directory and return filename -> full path."""
    index = {}
    for root, dirs, files in os.walk(data_dir):
        for fname in files:
            if fname.endswith(".abf"):
                index[fname] = os.path.join(root, fname)
    return index


def load_epoch(file_path, start_min, duration_min=0.5):
    """Load a single epoch from an ABF file without loading the full recording."""
    abf = pyabf.ABF(file_path)
    fs = abf.dataRate
    start_idx = int(start_min * 60 * fs)
    end_idx = int((start_min + duration_min) * 60 * fs)
    abf.setSweep(0)
    epoch = abf.sweepY[start_idx:end_idx].copy()
    del abf
    gc.collect()
    return epoch, fs


def get_per_animal_band_power(data_dir, manifest_filename):
    """
    Compute mean band power per animal from epoch manifest.
    Uses nperseg=fs for 1 Hz frequency resolution.
    Animal ID = first 5 characters of filename.
    """
    manifest = pd.read_excel(os.path.join(data_dir, manifest_filename))
    file_index = build_file_index(data_dir)
    animal_powers = {}

    for _, row in manifest.iterrows():
        fname = row["File"]
        if fname not in file_index:
            continue
        start_times = [
            float(t.strip())
            for t in str(row["Start_Times"]).split(",")
            if t.strip()
        ]
        animal_id = fname[:5]
        if animal_id not in animal_powers:
            animal_powers[animal_id] = []

        for start_min in start_times:
            try:
                epoch, fs = load_epoch(file_index[fname], start_min)
                freq, psd = compute_psd(epoch, fs, nperseg=int(fs))
                bp = compute_band_power(freq, psd)
                animal_powers[animal_id].append(bp)
                del epoch
                gc.collect()
            except Exception:
                pass

    rows = []
    for animal_id, bp_list in animal_powers.items():
        if bp_list:
            mean_bp = {k: np.mean([b[k] for b in bp_list]) for k in bp_list[0]}
            mean_bp["animal_id"] = animal_id
            rows.append(mean_bp)
    return pd.DataFrame(rows)


def process_full_recordings(data_dir):
    """
    Scan all ABF files in a directory and detect SADs across each full recording.
    Used for KA recordings where no epoch manifest is needed.
    """
    rows = []
    abf_files = [f for f in os.listdir(data_dir) if f.endswith(".abf")]
    print(f"  Found {len(abf_files)} ABF files in {os.path.basename(data_dir)}")

    for fname in abf_files:
        fpath = os.path.join(data_dir, fname)
        try:
            time, voltage, fs = load_abf(fpath)
            time, voltage = remove_artifacts(time, voltage)
            baseline = estimate_baseline(voltage, fs)
            lower_threshold = 2.0 * baseline
            events = detect_discharges(time, voltage, fs, lower_threshold=lower_threshold)
            duration_min = len(time) / fs / 60
            rows.append({
                "file": fname,
                "animal_id": fname[:5],
                "n_events": len(events),
                "rate_per_min": len(events) / duration_min if duration_min > 0 else 0,
                "mean_voltage_mV": events["voltage_mV"].mean() if len(events) > 0 else 0,
                "std_voltage_mV": events["voltage_mV"].std() if len(events) > 0 else 0,
                "mean_prominence": events["prominence"].mean() if len(events) > 0 else 0,
                "duration_min": duration_min,
                "baseline_mV": baseline,
            })
        except Exception as e:
            print(f"    ERROR — {fname}: {e}")
    return pd.DataFrame(rows)


print("Helper functions defined.")

## 3. SAD detection — 4-month kainic acid recordings

Full recordings scanned automatically using adaptive amplitude thresholding (2× baseline).

**Exclusion:** Animal 21507 excluded due to body weight loss >20% during the recording period, indicating illness unrelated to genotype.

In [ ]:
print("Processing WT KA recordings...")
sad_wt_raw = process_full_recordings(PATHS["4m_KA"]["WT"])

print("\nProcessing KO KA recordings...")
sad_ko = process_full_recordings(PATHS["4m_KA"]["KO"])

# Exclude animal 21507 — body weight loss >20%
sad_wt = sad_wt_raw[~sad_wt_raw["animal_id"].str.startswith("21507")].reset_index(drop=True)

# Per-animal means (2 recordings per animal)
per_animal_wt = sad_wt.groupby("animal_id")["rate_per_min"].mean().reset_index()
per_animal_ko = sad_ko.groupby("animal_id")["rate_per_min"].mean().reset_index()

print(f"\nWT animals (after exclusion): {len(per_animal_wt)}")
print(f"KO animals: {len(per_animal_ko)}")

### 3a. Example EEG trace with detected discharges

In [ ]:
ko_files = [f for f in os.listdir(PATHS["4m_KA"]["KO"]) if f.endswith(".abf")]
time, voltage, fs = load_abf(os.path.join(PATHS["4m_KA"]["KO"], ko_files[0]))
time, voltage = remove_artifacts(time, voltage)
baseline = estimate_baseline(voltage, fs)
events = detect_discharges(time, voltage, fs, lower_threshold=2.0 * baseline)

t_plot = time[:3600]
v_plot = voltage[:3600]
ev_plot = events[events["time_s"] <= t_plot[-1]]

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(t_plot, v_plot, color="#2C2C2A", lw=0.4, label="EEG (KO)")
ax.scatter(ev_plot["time_s"], ev_plot["voltage_mV"],
           color="#D85A30", s=18, zorder=5,
           label=f"SADs (n={len(ev_plot)} in first hour)")
ax.axhline(2.0 * baseline, color="#D85A30", lw=0.8, ls="--", label="Detection threshold")
ax.axhline(baseline, color="#378ADD", lw=0.8, ls="--", label="Baseline")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Voltage (mV)")
ax.set_title(f"Hippocampal EEG — KO mouse, 4 months post kainic acid")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "eeg_trace_ko_4m.png"), dpi=150, bbox_inches="tight")
plt.show()

### 3b. Discharge rate: WT vs KO (per animal)

In [ ]:
stat, pval = mannwhitneyu(
    per_animal_wt["rate_per_min"],
    per_animal_ko["rate_per_min"],
    alternative="two-sided"
)
sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"

fig, ax = plt.subplots(figsize=(4.5, 5))
colors = {"WT": "#378ADD", "KO": "#D85A30"}
for label, df_g in [("WT", per_animal_wt), ("KO", per_animal_ko)]:
    ax.scatter([label] * len(df_g), df_g["rate_per_min"],
               color=colors[label], alpha=0.7, s=60, zorder=3)
    ax.errorbar(label, df_g["rate_per_min"].mean(),
                yerr=df_g["rate_per_min"].sem(),
                fmt="_", markersize=28, markeredgewidth=2.5,
                color=colors[label], capsize=5, capthick=2)
ax.annotate(f"{sig}  p = {pval:.3f}", xy=(0.5, 0.93),
            xycoords="axes fraction", ha="center", fontsize=11)
ax.set_ylabel("Discharge rate (events / min, mean ± SEM)")
ax.set_title(f"SAD rate — 4 months post kainic acid\nWT n={len(per_animal_wt)} | KO n={len(per_animal_ko)}")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "sad_rate_per_animal.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"WT  mean ± SEM: {per_animal_wt['rate_per_min'].mean():.2f} ± {per_animal_wt['rate_per_min'].sem():.2f} events/min")
print(f"KO  mean ± SEM: {per_animal_ko['rate_per_min'].mean():.2f} ± {per_animal_ko['rate_per_min'].sem():.2f} events/min")
print(f"Mann-Whitney U: stat={stat:.1f}, p={pval:.4f} ({sig})")

## 4. Longitudinal band power analysis — 3, 6, and 12 months

Per-animal mean band power computed from baseline epoch recordings.
Uses 1 Hz frequency resolution (nperseg = sampling rate).
Statistical unit = one animal (epochs averaged per animal before comparison).

In [ ]:
print("Computing per-animal band power at 3m...")
bp_wt_3m = get_per_animal_band_power(PATHS["3m"]["WT"], PATHS["3m"]["manifest_wt"])
bp_ko_3m = get_per_animal_band_power(PATHS["3m"]["KO"], PATHS["3m"]["manifest_ko"])
print(f"  WT: {len(bp_wt_3m)} animals | KO: {len(bp_ko_3m)} animals")

print("\nComputing per-animal band power at 6m...")
bp_wt_6m = get_per_animal_band_power(PATHS["6m"]["WT"], PATHS["6m"]["manifest_wt"])
bp_ko_6m = get_per_animal_band_power(PATHS["6m"]["KO"], PATHS["6m"]["manifest_ko"])
print(f"  WT: {len(bp_wt_6m)} animals | KO: {len(bp_ko_6m)} animals")

print("\nComputing per-animal band power at 12m...")
bp_wt_12m = get_per_animal_band_power(PATHS["12m"]["WT"], PATHS["12m"]["manifest_wt"])
bp_ko_12m = get_per_animal_band_power(PATHS["12m"]["KO"], PATHS["12m"]["manifest_ko"])
print(f"  WT: {len(bp_wt_12m)} animals | KO: {len(bp_ko_12m)} animals")

### 4a. Full band power comparison at 12 months

In [ ]:
bands = ["delta", "theta", "alpha", "beta", "gamma"]
band_ranges = ["0.5-4Hz", "4-8Hz", "8-13Hz", "13-30Hz", "30-50Hz"]
colors = {"WT": "#378ADD", "KO": "#D85A30"}

print(f"{'Band':<10} {'WT mean±SEM':<22} {'KO mean±SEM':<22} {'p-value':<12} {'sig'}")
print("-" * 72)

pvals_12m = {}
for band in bands:
    stat, pval = mannwhitneyu(
        bp_wt_12m[band], bp_ko_12m[band], alternative="two-sided"
    )
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    pvals_12m[band] = (pval, sig)
    wt_str = f"{bp_wt_12m[band].mean():.4f}±{bp_wt_12m[band].sem():.4f}"
    ko_str = f"{bp_ko_12m[band].mean():.4f}±{bp_ko_12m[band].sem():.4f}"
    print(f"{band:<10} {wt_str:<22} {ko_str:<22} {pval:<12.4f} {sig}")

fig, axes = plt.subplots(1, 5, figsize=(16, 5))
for ax, band, band_range in zip(axes, bands, band_ranges):
    for label, df_g in [("WT", bp_wt_12m), ("KO", bp_ko_12m)]:
        ax.scatter([label] * len(df_g), df_g[band],
                   color=colors[label], alpha=0.6, s=40, zorder=3)
        ax.errorbar(label, df_g[band].mean(), yerr=df_g[band].sem(),
                    fmt="_", markersize=24, markeredgewidth=2.5,
                    color=colors[label], capsize=4, capthick=2)
    pval, sig = pvals_12m[band]
    ax.annotate(f"{sig}\np={pval:.3f}", xy=(0.5, 0.93),
                xycoords="axes fraction", ha="center", fontsize=9)
    ax.set_title(f"{band.capitalize()}\n{band_range}", fontsize=10)
    if ax == axes[0]:
        ax.set_ylabel("Normalized power")

fig.suptitle(
    f"Band power at 12 months — WT (n={len(bp_wt_12m)}) vs C9orf72-KO (n={len(bp_ko_12m)})",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "band_power_12m_all_bands.png"), dpi=150, bbox_inches="tight")
plt.show()

### 4b. Longitudinal theta power progression

Theta band (4–8 Hz) shows a U-shaped divergence pattern:
elevated in KO at 3m, converges at 6m, re-emerges at 12m.

In [ ]:
timepoints_labels = ["3 months", "6 months", "12 months"]
wt_dfs = [bp_wt_3m, bp_wt_6m, bp_wt_12m]
ko_dfs = [bp_ko_3m, bp_ko_6m, bp_ko_12m]

wt_means = [df["theta"].mean() for df in wt_dfs]
wt_sems  = [df["theta"].sem()  for df in wt_dfs]
ko_means = [df["theta"].mean() for df in ko_dfs]
ko_sems  = [df["theta"].sem()  for df in ko_dfs]

pvals_theta, sigs_theta = [], []
for wt_df, ko_df in zip(wt_dfs, ko_dfs):
    stat, pval = mannwhitneyu(wt_df["theta"], ko_df["theta"], alternative="two-sided")
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "ns"
    pvals_theta.append(pval)
    sigs_theta.append(sig)

x = np.arange(len(timepoints_labels))
fig, ax = plt.subplots(figsize=(7, 5))

ax.errorbar(x, wt_means, yerr=wt_sems,
            color="#378ADD", lw=2, marker="o", markersize=8,
            capsize=5, capthick=2, label="WT")
ax.errorbar(x, ko_means, yerr=ko_sems,
            color="#D85A30", lw=2, marker="o", markersize=8,
            capsize=5, capthick=2, label="KO")

for i, (wt_df, ko_df) in enumerate(zip(wt_dfs, ko_dfs)):
    ax.scatter(np.full(len(wt_df), i) + 0.05, wt_df["theta"],
               color="#378ADD", alpha=0.3, s=30, zorder=2)
    ax.scatter(np.full(len(ko_df), i) - 0.05, ko_df["theta"],
               color="#D85A30", alpha=0.3, s=30, zorder=2)

for i, (sig, pval) in enumerate(zip(sigs_theta, pvals_theta)):
    y_max = max(wt_dfs[i]["theta"].max(), ko_dfs[i]["theta"].max()) + 0.02
    ax.annotate(f"{sig}\np={pval:.3f}", xy=(i, y_max), ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels([
    f"{label}\nWT n={len(wt_df)} | KO n={len(ko_df)}"
    for label, wt_df, ko_df in zip(timepoints_labels, wt_dfs, ko_dfs)
])
ax.set_ylabel("Theta band power (4–8 Hz, normalized)", fontsize=11)
ax.set_title("Longitudinal theta power — WT vs C9orf72-KO", fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "theta_power_longitudinal.png"), dpi=150, bbox_inches="tight")
plt.show()

print("Theta power progression:")
for tp, wt_df, ko_df, pval, sig in zip(
    ["3m", "6m", "12m"], wt_dfs, ko_dfs, pvals_theta, sigs_theta
):
    ratio = ko_df["theta"].mean() / wt_df["theta"].mean()
    print(f"  {tp}: WT={wt_df['theta'].mean():.4f} | KO={ko_df['theta'].mean():.4f} | KO/WT={ratio:.3f} | p={pval:.4f} ({sig})")

## 5. WT vs KO classifier — 4-month SAD features

Random Forest classifier trained on per-recording discharge features.
Stratified 5-fold cross-validation. `class_weight='balanced'` handles unequal group sizes.

In [ ]:
# Build feature matrix from per-recording summaries
sad_wt_feats = sad_wt.copy()
sad_ko_feats = sad_ko.copy()

X, y = build_feature_matrix(sad_wt_feats, sad_ko_feats)
print(f"Feature matrix: {X.shape[0]} recordings x {X.shape[1]} features")
print(f"Features: {list(X.columns)}")
print(f"Class balance — WT: {(y==0).sum()}, KO: {(y==1).sum()}")

In [ ]:
pipeline, X_test, y_test = train_classifier(X, y)
metrics = evaluate_classifier(pipeline, X_test, y_test)
plot_roc_and_pr(metrics, save_path=os.path.join(FIGURES_DIR, "roc_pr_curves.png"))

In [ ]:
plot_feature_importance(
    pipeline,
    feature_names=list(X.columns),
    save_path=os.path.join(FIGURES_DIR, "feature_importance.png")
)

rf = pipeline.named_steps["clf"]
importances = rf.feature_importances_
ranked = sorted(zip(list(X.columns), importances), key=lambda x: x[1], reverse=True)
print("\nFeature importance ranking:")
for name, imp in ranked:
    print(f"  {name}: {imp:.4f}")

## 6. Key findings

| Metric | WT | KO |
|--------|----|----|
| SAD rate — 4m (events/min, mean ± SEM) | 7.84 ± 2.72 | 8.53 ± 2.19 |
| Mann-Whitney p-value (SAD rate) | ns (p=0.950) | — |
| Theta power — 3m | 0.261 | 0.296 |
| Theta power — 6m | 0.277 | 0.287 |
| Theta power — 12m | 0.305 | 0.329 |
| Theta KO/WT ratio at 3m | 1.132 (p=0.021*) | — |
| Theta KO/WT ratio at 6m | 1.038 (ns) | — |
| Theta KO/WT ratio at 12m | 1.077 (p=0.030*) | — |
| Classifier ROC-AUC | 0.850 | — |
| Top discriminative feature | discharge rate | — |

**Biological interpretation:**
C9orf72-KO mice show no significant difference in kainic acid-induced seizure rate at 4 months, suggesting C9orf72 loss does not affect acute seizure susceptibility. However, longitudinal EEG analysis reveals a U-shaped elevation in theta band power (4–8 Hz) in KO mice — significant at 3 months (KO/WT=1.13, p=0.021) and 12 months (KO/WT=1.08, p=0.030), with convergence at 6 months. This pattern suggests early network dysfunction, transient compensation, and re-emergence of dysfunction at late disease stage — consistent with the progressive nature of ALS/FTD pathology.

---
*Animals: WT n=6–19 per timepoint | KO n=8–13 per timepoint (varies by timepoint)*  
*Statistical test: Mann-Whitney U (two-sided), appropriate for non-normal distributions*  
*Exclusion: Animal 21507 excluded — body weight loss >20% indicating illness unrelated to genotype*